In [1]:
import subprocess
from pathlib import Path
import cftime

## Configuration

In [2]:
# Input folder and settings
casename = "b.e13.HF-TNST.rcp85.ne120_t12.1920-2100.010"
data_dir = "/glade/campaign/univ/utam0017/MOAAP"
print(f"{data_dir}/{casename}")

date_begin = cftime.DatetimeNoLeap(2080, 1, 1, 0)
date_final = cftime.DatetimeNoLeap(2101, 1, 1, 0)  # Exclusive

dt_hrs_data = 1
dt_stride_tracking = 1  # Convert hourly to 3-hourly before tracking?
months_per_job = 4  # Has to be integer

# Python wrapper to run MOAAP
cesm_moaap_wrapper = Path("./cesm_moaap_wrapper.py").resolve()

# Output folder
out_dir = Path("/glade/derecho/scratch/jiangzhu/moaap")

job_script_dir = Path("./pbs_jobs")
job_script_dir.mkdir(parents=True, exist_ok=True)

/glade/campaign/univ/utam0017/MOAAP/b.e13.HF-TNST.rcp85.ne120_t12.1920-2100.010


In [3]:

track_all = False
track_mcs = False
track_mcs_tc_ar = True

if track_all:
    varlist = ["PSL", "Z500", "PRECT", "FLUT", "U850", "V850", "T850",
               "uIVT", "vIVT", "Q850", "U200", "V200"]

if track_mcs:
    varlist = ["PRECT", "FLUT"]

if track_mcs_tc_ar:
    varlist = ["PSL", "FLUT", "T850", "PRECT", "uIVT", "vIVT", "U850", "V850"]

## Helper: advance months & write job scripts

In [4]:
def add_months_noleap(t: cftime.DatetimeNoLeap, n: int) -> cftime.DatetimeNoLeap:
    total = (t.month - 1) + n
    y = t.year + total // 12
    m = total % 12 + 1

    # noleap month lengths
    mdays = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31][m - 1]
    d = min(t.day, mdays)

    return cftime.DatetimeNoLeap(y, m, d, t.hour, t.minute, t.second)


def write_job_script(job_path: Path, yyyy: int, mm: int, varlist):
    tag = f"{yyyy:04d}{mm:02d}"

    # One quoted CSV string, e.g. "PSL,Z500,PRECT"
    var_csv = ",".join(v.strip() for v in varlist if v and str(v).strip())

    script = f"""#!/bin/bash -l
#PBS -N moaap_{tag}
#PBS -A P93300324
#PBS -l select=1:ncpus=12:mem=720GB
#PBS -l walltime=24:00:00
#PBS -q casper
#PBS -j oe

module load conda
conda activate npl

python "{cesm_moaap_wrapper}" \\
  --data-dir "{data_dir}" \\
  --casename "{casename}" \\
  --out-dir "{out_dir}" \\
  --year {yyyy} \\
  --month {mm} \\
  --months-per-job {months_per_job} \\
  --dt-hrs-data {dt_hrs_data} \\
  --dt-stride-tracking {dt_stride_tracking} \\
  --varlist "{var_csv}"
"""
    job_path.write_text(script)
    job_path.chmod(0o750)
    return job_path


# Optional: test one case inside JupyterHub (no qsub)
def test_one(yyyy: int, mm: int, varlist):
    import importlib.util

    spec = importlib.util.spec_from_file_location("cesm_moaap_tracking", cesm_moaap_wrapper)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)

    var_csv = ",".join(v.strip() for v in varlist if v and str(v).strip())

    mod.main([
        "--data-dir", data_dir,
        "--casename", casename,
        "--out-dir", str(out_dir),
        "--year", str(yyyy),
        "--month", str(mm),
        "--months-per-job", str(months_per_job),
        "--dt-dt_hrs_data-data", str(dt_hrs_data),
        "--dt-stride-tracking", str(dt_stride_tracking),
        "--varlist", var_csv,
    ])

## Main loop to submit multiple jobs

In [5]:
current = date_begin
while current < date_final:
    yyyy, mm = current.year, current.month

    job_path = job_script_dir / f"casper_submit_{yyyy:04d}{mm:02d}_{dt_stride_tracking}hourly.sh"
    write_job_script(job_path, yyyy, mm, varlist)

    jobid = subprocess.check_output(["qsub", str(job_path)], text=True).strip()
    current = add_months_noleap(current, months_per_job)
    print(f"Submitted job for {yyyy}-{mm:02d}, {current.year}-{current.month:02d} -> {jobid}")

Submitted job for 2080-01, 2080-05 -> 2612047.casper-pbs
Submitted job for 2080-05, 2080-09 -> 2612048.casper-pbs
Submitted job for 2080-09, 2081-01 -> 2612049.casper-pbs
Submitted job for 2081-01, 2081-05 -> 2612050.casper-pbs
Submitted job for 2081-05, 2081-09 -> 2612051.casper-pbs
Submitted job for 2081-09, 2082-01 -> 2612052.casper-pbs
Submitted job for 2082-01, 2082-05 -> 2612053.casper-pbs
Submitted job for 2082-05, 2082-09 -> 2612054.casper-pbs
Submitted job for 2082-09, 2083-01 -> 2612055.casper-pbs
Submitted job for 2083-01, 2083-05 -> 2612056.casper-pbs
Submitted job for 2083-05, 2083-09 -> 2612057.casper-pbs
Submitted job for 2083-09, 2084-01 -> 2612058.casper-pbs
Submitted job for 2084-01, 2084-05 -> 2612059.casper-pbs
Submitted job for 2084-05, 2084-09 -> 2612060.casper-pbs
Submitted job for 2084-09, 2085-01 -> 2612061.casper-pbs
Submitted job for 2085-01, 2085-05 -> 2612062.casper-pbs
Submitted job for 2085-05, 2085-09 -> 2612063.casper-pbs
Submitted job for 2085-09, 2086